# VictoriaMetrics Anomaly Detection mit Facebook Prophet

Erkennt Anomalien in VictoriaMetrics-Metriken mithilfe von **Facebook Prophet**.

**Ablauf:**
1. Konfiguration laden
2. Historische Daten aus VictoriaMetrics laden (Training)
3. Prophet-Modell trainieren
4. Aktuelle Daten auswerten (Anomalie-Erkennung)
5. Ergebnisse zurück nach VictoriaMetrics schreiben

## 1. Konfiguration

Passe die YAML-Konfiguration und `DRY_RUN` nach Bedarf an.

In [ ]:
import io
import yaml

CONFIG_YAML = """
# ============================================================
# VictoriaMetrics Anomaly Detection – Konfigurationsdatei
# ============================================================

# VictoriaMetrics Verbindung (Cluster-Modus: getrennte URLs für Lesen/Schreiben)
victoriametrics:
  read_url: "http://vmselect:8481/select/0/prometheus"   # vmselect
  write_url: "http://vminsert:8480/insert/0/prometheus"  # vminsert
  timeout: 30                      # HTTP-Timeout in Sekunden

# Anomalie-Erkennung: Abfrage-Konfiguration
queries:
  - name: "system_load5"
    promql: "system_load5{host='ubuntu'}"
    description: "System Load"

  - name: "mem_used_percent{host='ubuntu'}"
    promql: "mem_used_percent"
    description: "Memory-Auslastung in %"

  - name: "netstat_tcp_established"
    promql: "netstat_tcp_established{host='ubuntu'}"
    description: "netstat_tcp_established"

  - name: "network_traffic_in"
    promql: 'rate(net_bytes_recv{interface=~"wlp1s0|eth0|wlp4s0",host="ubuntu"})'
    description: "Netzwerk-Eingangsrate"

  - name: "network_traffic_sent"
    promql: 'rate(net_bytes_sent{interface=~"wlp1s0|eth0|wlp4s0",host="ubuntu"})'
    description: "Netzwerk-Ausgangsrate"

# Prophet-Modell Konfiguration
model:
  # Konfidenzintervall: 0.95 = 95% → engeres Band, mehr Anomalien
  #                    0.99 = 99% → weiteres Band, weniger Anomalien
  interval_width: 0.99
  yearly_seasonality: false        # Nur sinnvoll bei > 1 Jahr Daten
  weekly_seasonality: true
  daily_seasonality: true
  changepoint_prior_scale: 0.05   # Sensitivität für Trendwechsel
  seasonality_prior_scale: 10.0   # Sensitivität für Saisonalität
  training_lookback: "7d"         # Wie weit zurück Daten geladen werden
  forecast_horizon: "1h"

# Scheduling
scheduler:
  interval_minutes: 5
  start: "now"

# Alerting
alerting:
  enabled: true
  zscore_threshold: 3.0
  webhook:
    enabled: false
    url: "http://alertmanager:9093/api/v1/alerts"
    timeout: 10
  log_anomalies: true
  log_file: "anomalies.log"

# Ergebnisse zurück nach VictoriaMetrics schreiben
writer:
  enabled: true
  metric_prefix: "vmanomaly"

# Logging
logging:
  level: "INFO"
  format: "%(asctime)s [%(levelname)s] %(message)s"
  file: "vmanomaly.log"
"""

config = yaml.safe_load(io.StringIO(CONFIG_YAML))

# Notebook-Option: True = kein Schreiben nach VictoriaMetrics
DRY_RUN = True

## 2. Imports & Logging-Konfiguration

In [ ]:
import json
import logging
import math
import time
from datetime import datetime, timedelta
from typing import Optional

import numpy as np
import pandas as pd
import requests
from prophet import Prophet

import warnings
warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

## 3. Hilfsfunktionen

In [ ]:
def parse_duration(duration_str: str) -> timedelta:
    """Parst Dauer-Strings wie '7d', '2h', '30m', '60s'."""
    units = {"s": 1, "m": 60, "h": 3600, "d": 86400, "w": 604800}
    unit = duration_str[-1].lower()
    value = int(duration_str[:-1])
    if unit not in units:
        raise ValueError(f"Unbekannte Zeiteinheit: {unit}")
    return timedelta(seconds=value * units[unit])


def setup_logging(config: dict):
    """Konfiguriert das Logging."""
    log_config = config.get("logging", {})
    level = getattr(logging, log_config.get("level", "INFO").upper(), logging.INFO)
    fmt = log_config.get("format", "%(asctime)s [%(levelname)s] %(message)s")
    log_file = log_config.get("file", None)

    handlers = [logging.StreamHandler()]
    if log_file:
        handlers.append(logging.FileHandler(log_file))

    # force=True überschreibt ggf. bereits vorhandene Jupyter-Logging-Konfiguration
    logging.basicConfig(level=level, format=fmt, handlers=handlers, force=True)


def _label_fingerprint(labels: dict) -> str:
    """Eindeutiger Schlüssel für eine Label-Kombination (stabile Sortierung)."""
    return json.dumps(labels, sort_keys=True)


def _labels_to_promstr(labels: dict) -> str:
    """Label-Dict → Prometheus-Expositions-String  key="val",... (sortiert)."""
    return ",".join(f'{k}="{v}"' for k, v in sorted(labels.items()))

## 4. VictoriaMetrics Client

In [ ]:
class VMClient:
    """Kommuniziert mit der VictoriaMetrics HTTP API."""

    def __init__(self, config: dict):
        vm = config["victoriametrics"]
        self.read_url  = vm["read_url"].rstrip("/")
        self.write_url = vm["write_url"].rstrip("/")
        self.timeout   = vm.get("timeout", 30)

    def query_range(
        self,
        promql: str,
        start: datetime,
        end: datetime,
        step: str = "5m"
    ) -> list:
        """
        Führt eine range query durch.
        Gibt eine Liste von (labels, df) zurück – ein Eintrag pro Zeitreihe.
        df-Spalten: ds_epoch (float, Unix), ds (datetime, naive UTC), y (float)
        """
        import calendar
        url = f"{self.read_url}/api/v1/query_range"
        params = {
            "query": promql,
            "start": calendar.timegm(start.timetuple()),
            "end":   calendar.timegm(end.timetuple()),
            "step":  step,
        }
        try:
            resp = requests.get(url, params=params, timeout=self.timeout)
            resp.raise_for_status()
            data = resp.json()
        except requests.RequestException as e:
            logging.error(f"VictoriaMetrics-Abfrage fehlgeschlagen: {e}")
            return []

        if data.get("status") != "success":
            logging.error(f"API-Fehler: {data}")
            return []

        results = data.get("data", {}).get("result", [])
        if not results:
            logging.warning(f"Keine Daten für Query: {promql}")
            return []

        series_list = []
        for series in results:
            labels = series.get("metric", {})
            rows = []
            for ts, val in series["values"]:
                try:
                    rows.append({
                        "ds_epoch": float(ts),
                        "ds": datetime(1970, 1, 1) + timedelta(seconds=float(ts)),
                        "y": float(val),
                    })
                except (ValueError, TypeError):
                    continue
            if rows:
                df = (pd.DataFrame(rows)
                        .sort_values("ds_epoch")
                        .reset_index(drop=True))
                series_list.append((labels, df))

        return series_list

    def write_batch(self, lines: list):
        """Schreibt mehrere Metriken auf einmal via Prometheus remote write."""
        url = f"{self.write_url}/api/v1/import/prometheus"
        payload = "\n".join(lines)
        try:
            resp = requests.post(
                url,
                data=payload,
                headers={"Content-Type": "text/plain"},
                timeout=self.timeout,
            )
            resp.raise_for_status()
        except requests.RequestException as e:
            logging.error(f"Batch-Schreiben fehlgeschlagen: {e}")

## 5. Prophet Anomalie-Erkennung

In [ ]:
class ProphetAnomalyDetector:
    """
    Trainiert ein Prophet-Modell auf historischen Daten und erkennt
    Anomalien im aktuellen Zeitfenster.
    """

    def __init__(self, model_config: dict):
        self.interval_width        = model_config.get("interval_width", 0.99)
        self.yearly_seasonality    = model_config.get("yearly_seasonality", False)
        self.weekly_seasonality    = model_config.get("weekly_seasonality", True)
        self.daily_seasonality     = model_config.get("daily_seasonality", True)
        self.changepoint_prior_scale = model_config.get("changepoint_prior_scale", 0.05)
        self.seasonality_prior_scale = model_config.get("seasonality_prior_scale", 10.0)
        self.zscore_threshold      = 3.0
        self.model: Optional[Prophet] = None

    def train(self, df: pd.DataFrame) -> bool:
        """Trainiert das Prophet-Modell. Erwartet Spalten: ds (datetime), y (float)."""
        if df.empty or len(df) < 10:
            logging.warning("Zu wenige Datenpunkte für Prophet-Training (min. 10).")
            return False

        df = df.replace([np.inf, -np.inf], np.nan).dropna()
        if df.empty:
            logging.warning("DataFrame nach Bereinigung leer.")
            return False

        try:
            self.model = Prophet(
                interval_width=self.interval_width,
                yearly_seasonality=self.yearly_seasonality,
                weekly_seasonality=self.weekly_seasonality,
                daily_seasonality=self.daily_seasonality,
                changepoint_prior_scale=self.changepoint_prior_scale,
                seasonality_prior_scale=self.seasonality_prior_scale,
            )
            self.model.fit(df)
            logging.info(f"Prophet-Modell trainiert auf {len(df)} Datenpunkten.")
            return True
        except Exception as e:
            logging.error(f"Prophet-Training fehlgeschlagen: {e}")
            return False

    def predict(self, df: pd.DataFrame) -> pd.DataFrame:
        """Erstellt Prognosen und gibt DataFrame mit Anomalie-Flags zurück."""
        if self.model is None:
            raise RuntimeError("Modell wurde noch nicht trainiert.")
        if df.empty:
            return pd.DataFrame()

        forecast = self.model.predict(df[["ds"]].copy())
        result = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
        merge_cols = ["ds", "ds_epoch", "y"] if "ds_epoch" in df.columns else ["ds", "y"]
        result = result.merge(df[merge_cols], on="ds", how="left")

        result["anomaly"] = (
            (result["y"] < result["yhat_lower"]) |
            (result["y"] > result["yhat_upper"])
        )
        result["deviation"] = result["y"] - result["yhat"]

        std = result["deviation"].std()
        result["zscore"] = (result["deviation"] / std).abs() if std > 0 else 0.0
        result["anomaly_score"] = result["zscore"].apply(lambda z: min(1.0, z / 10.0))
        result["anomaly_zscore"] = result["zscore"] > self.zscore_threshold
        return result

    def detect(self, train_df: pd.DataFrame, eval_df: pd.DataFrame) -> pd.DataFrame:
        """Training + Vorhersage in einem Schritt."""
        if not self.train(train_df):
            return pd.DataFrame()
        return self.predict(eval_df)

## 6. Alert-Manager

In [ ]:
class AlertManager:
    """Verwaltet und versendet Anomalie-Alerts."""

    def __init__(self, config: dict):
        self.alert_config  = config.get("alerting", {})
        self.enabled       = self.alert_config.get("enabled", True)
        self.log_anomalies = self.alert_config.get("log_anomalies", True)
        self.log_file      = self.alert_config.get("log_file", "anomalies.log")
        self.webhook_config = self.alert_config.get("webhook", {})

    def process(self, query_name: str, result: pd.DataFrame):
        if not self.enabled or result.empty:
            return
        anomalies = result[result["anomaly"] == True]
        for _, row in anomalies.iterrows():
            self._handle_anomaly(query_name, row)

    def _handle_anomaly(self, query_name: str, row):
        msg = (
            f"[ANOMALIE] {query_name} | "
            f"Zeit: {row['ds']} | "
            f"Wert: {row['y']:.4f} | "
            f"Erwartet: {row['yhat']:.4f} | "
            f"Intervall: [{row['yhat_lower']:.4f}, {row['yhat_upper']:.4f}] | "
            f"Score: {row.get('anomaly_score', 0):.3f}"
        )
        logging.warning(msg)
        if self.log_anomalies:
            self._write_log(msg)
        if self.webhook_config.get("enabled", False):
            self._send_webhook(query_name, row)

    def _write_log(self, msg: str):
        try:
            with open(self.log_file, "a") as f:
                f.write(msg + "\n")
        except IOError as e:
            logging.error(f"Log-Schreiben fehlgeschlagen: {e}")

    def _send_webhook(self, query_name: str, row):
        payload = [{
            "labels": {"alertname": "VMAnomaly", "query": query_name, "severity": "warning"},
            "annotations": {
                "summary": f"Anomalie in {query_name}",
                "description": (
                    f"Wert {row['y']:.4f} außerhalb des erwarteten Bereichs "
                    f"[{row['yhat_lower']:.4f}, {row['yhat_upper']:.4f}]"
                ),
            },
        }]
        try:
            requests.post(
                self.webhook_config["url"],
                json=payload,
                timeout=self.webhook_config.get("timeout", 10),
            )
        except requests.RequestException as e:
            logging.error(f"Webhook-Versand fehlgeschlagen: {e}")

## 7. Hauptklasse: VMAnomalyDetection

In [ ]:
class VMAnomalyDetection:
    """Orchestriert VMClient, ProphetAnomalyDetector und AlertManager."""

    def __init__(self, config: dict, dry_run: bool = False):
        self.config        = config
        self.dry_run       = dry_run
        self.vm            = VMClient(config)
        self.alert_manager = AlertManager(config)
        self.model_config  = config.get("model", {})
        self.writer_config = config.get("writer", {})
        self.queries       = config.get("queries", [])
        self.training_lookback = parse_duration(
            self.model_config.get("training_lookback", "7d")
        )

    def run_once(self):
        """Führt einen einzelnen Erkennungs-Durchlauf durch."""
        now_epoch = time.time()
        now       = datetime(1970, 1, 1) + timedelta(seconds=now_epoch)
        train_start = now - self.training_lookback
        train_end   = now - timedelta(minutes=5)
        eval_start  = train_end
        eval_end    = now

        logging.info(
            f"Starte Anomalie-Erkennung | "
            f"Training: {train_start:%Y-%m-%d %H:%M} → {train_end:%H:%M} | "
            f"Eval: {eval_start:%H:%M} → {eval_end:%H:%M}"
        )

        output_lines = []

        for query_cfg in self.queries:
            name   = query_cfg["name"]
            promql = query_cfg["promql"]
            desc   = query_cfg.get("description", name)
            logging.info(f"Verarbeite: {desc} ({name})")

            # Jede Query liefert eine Liste von (labels, df) – eine pro Zeitreihe
            train_series = self.vm.query_range(promql, train_start, train_end)
            if not train_series:
                logging.warning(f"Keine Trainingsdaten für '{name}' – überspringe.")
                continue

            eval_series = self.vm.query_range(promql, eval_start, eval_end)
            if not eval_series:
                logging.warning(f"Keine Eval-Daten für '{name}' – überspringe.")
                continue

            # Serien per Label-Fingerprint paaren
            train_by_fp = {_label_fingerprint(lbl): (lbl, df) for lbl, df in train_series}
            eval_by_fp  = {_label_fingerprint(lbl): (lbl, df) for lbl, df in eval_series}
            common_fps  = set(train_by_fp) & set(eval_by_fp)

            logging.info(
                f"  {len(train_series)} Train- / {len(eval_series)} Eval-Serien, "
                f"{len(common_fps)} gemeinsame."
            )

            for fp in common_fps:
                metric_labels, train_df = train_by_fp[fp]
                _,             eval_df  = eval_by_fp[fp]
                label_str = _labels_to_promstr(metric_labels)

                detector = ProphetAnomalyDetector(self.model_config)
                result   = detector.detect(train_df, eval_df)

                if result.empty:
                    continue

                n_anomalies = result["anomaly"].sum()
                logging.info(
                    f"  {{{label_str}}}: "
                    f"{n_anomalies}/{len(result)} Anomalien erkannt."
                )

                self.alert_manager.process(f"{name}{{{label_str}}}", result)

                if self.writer_config.get("enabled", True) and not self.dry_run:
                    prefix       = self.writer_config.get("metric_prefix", "vmanomaly")
                    write_labels = _labels_to_promstr({**metric_labels, "query": name})

                    for _, row in result.iterrows():
                        ts_ms = int(row.get("ds_epoch", 0) * 1000)
                        output_lines.append(
                            f'{prefix}_anomaly{{{write_labels}}} ' +
                            f'{1 if row["anomaly"] else 0} {ts_ms}'
                        )
                        score = row.get("anomaly_score", 0.0)
                        if not math.isnan(score):
                            output_lines.append(
                                f'{prefix}_score{{{write_labels}}} {score:.6f} {ts_ms}'
                            )
                        dev = row.get("deviation", 0.0)
                        if not math.isnan(dev):
                            output_lines.append(
                                f'{prefix}_deviation{{{write_labels}}} {dev:.6f} {ts_ms}'
                            )

        if output_lines and not self.dry_run:
            self.vm.write_batch(output_lines)
            logging.info(f"{len(output_lines)} Metriken nach VictoriaMetrics geschrieben.")
        elif self.dry_run:
            logging.info(f"[DRY-RUN] {len(output_lines)} Metriken würden geschrieben.")

    def run_scheduled(self, interval_minutes: int):
        """Periodische Ausführung – blockiert den Kernel bis zur Unterbrechung."""
        logging.info(f"Starte Scheduler – Intervall: {interval_minutes} Minuten")
        while True:
            try:
                self.run_once()
            except Exception as e:
                logging.error(f"Fehler im Hauptdurchlauf: {e}", exc_info=True)
            logging.info(f"Nächste Ausführung in {interval_minutes} Minuten...")
            time.sleep(interval_minutes * 60)

## 8. Ausführung

**Optionen:**
- `run_once()` – einmaliger Durchlauf *(Standard)*
- `run_scheduled(n)` – periodische Ausführung alle n Minuten (**blockiert den Kernel!**)

Setze `DRY_RUN = True` in Zelle 1, um das Zurückschreiben nach VictoriaMetrics zu überspringen.

In [ ]:
# Logging initialisieren
setup_logging(config)

logging.info("=" * 60)
logging.info("VictoriaMetrics Anomaly Detection gestartet")
logging.info(f"Dry-Run: {DRY_RUN}")
logging.info("=" * 60)

# Detektor initialisieren
detector = VMAnomalyDetection(config, dry_run=DRY_RUN)

# --- Einmalig ausführen ---
detector.run_once()

# --- Periodisch ausführen (blockiert Kernel bis Interrupt mit ■ Stop): ---
# interval = config.get("scheduler", {}).get("interval_minutes", 5)
# detector.run_scheduled(interval)